# Homologous recombination events

In [ ]:
# set working directory to project folder
import sys, os
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import plotly.graph_objects as go

import plotly.io as pio
pio.renderers.default = "browser" 

import altair as alt
from itertools import combinations
import numpy as np
import math
import pypangraph as pp
from Bio import Phylo, SeqIO, AlignIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

import random
import plotly.graph_objects as go
import plotly.express as px

import scipy.cluster.hierarchy as sch
from collections import defaultdict, deque, Counter
from Bio import Phylo

from pathlib import Path
import subprocess

from Bio.Phylo.TreeConstruction import DistanceCalculator
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster
from scipy.spatial.distance import squareform

from junction_analysis.helpers import get_tree_order, cluster_by_tree, strip_newick_suffixes, write_shared_nodes_fasta, simplify_cluster_keys, count_trna_per_junction, get_core_alignment_lengths, core_block_snp_gaps, add_singleton_gain_loss_isolate_columns, add_singleton_has_own_cluster
from junction_analysis.junction_trees import build_tree_from_block_list, compute_pairwise_distances, cluster_tree_by_branch_length
import junction_analysis.pangraph_utils as pu
from junction_analysis.plotting import plot_junction_pangraph_interactive, plot_block_distance_distribution, plot_pairwise_distance_hist, plot_event_counts_distribution, plot_events_per_junction, plot_cluster_count_distribution, plot_event_length_distribution, plot_cluster_count_vs_diversity, plot_marginal_scatter, plot_all_junctions_pairwise_distances, plot_snp_position_cdfs, plot_snp_gap_histogram, plot_core_alignment_lengths, plot_scatter
from junction_analysis.consensus import find_consensus_paths, make_deduplicated_paths, consensus_paths_and_assignments, find_consensus_paths_core, cluster_map_to_dataframe, collect_all_pairwise_distances, save_consensus_cache, collect_all_branch_lengths
from junction_analysis.block_alignment import create_block_msas, summarize_block_msas, analyze_alignment, cluster_alignment, retrieve_cluster_assignments
from junction_analysis.annotate_insertions import get_insertions_deletions_from_consensus, write_insertions_fasta, summarize_deletions_consensus, load_all_deletions_summaries, summarize_inversions_consensus, summarize_translocations_consensus, write_insertions_fasta, load_all_insertions_summaries, load_all_inversions_summaries, load_all_translocations_summaries, combine_all_junctions_summaries, count_events_per_junction, deduplicate_events, summarize_ambiguous_insertions, load_all_ambiguous_insertions_summaries

In [23]:
#jdf_all = add_singleton_gain_loss_isolate_columns(jdf_all, pangraph_dir="../results/junction_pangraphs")
#cluster_df = pd.read_csv("../results/consensus_analysis/cluster_maps_0_001.csv")
#jdf_all = add_singleton_has_own_cluster(jdf_all, cluster_df)
#jdf_all.to_csv("../results/junction_summary_df.csv", index=False)

jdf_all = pd.read_csv("../results/junction_summary_df.csv")
jdf_all

,junction_name,n_iso,n_blocks,has_dupl,n_categories,majority_category,singleton,cat_entropy,n_nodes,min_length,...,mean_length_translocation,mean_length_inversion,mean_event_length,block_ji_dist,edge_ji_dist,has_singleton_gain,has_singleton_loss,singleton_gain_isolate,singleton_loss_isolate,singleton_has_own_cluster
0,HUTOPWFGVH_f__WJBYSSJHSE_f,222,5,False,3,218,False,0.100354,11,18176,...,0.0,0.0,744.000000,0.991064,0.973218,NaN,NaN,NaN,NaN,NaN
1,IZQZNHJQRQ_f__ZVPGGEJIIF_f,222,3,False,2,221,True,0.028831,5,11739,...,0.0,0.0,777.000000,0.996997,0.990991,True,False,NZ_CP051615.1,NaN,False
2,JNLRGIQXOF_f__YQUHGDANHE_f,222,3,False,2,221,True,0.028831,5,27949,...,0.0,0.0,1437.000000,0.996997,0.990991,True,False,NZ_AP022362.1,NaN,False
3,JMOMDSHCBS_r__SPDPCYMYDN_r,222,3,False,2,220,False,0.051397,5,7364,...,0.0,0.0,777.000000,0.994021,0.982064,NaN,NaN,NaN,NaN,NaN
4,JKRDVEYDGL_f__OTPJRRWJRK_f,222,3,False,2,221,True,0.028831,5,10056,...,0.0,0.0,38175.000000,0.996997,0.990991,True,False,NZ_CP032201.1,NaN,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
513,PLTCZQCVRD_f__RYYAQMEJGY_f,3,48,True,3,1,False,1.098612,80,101503,...,3366.0,0.0,7118.870968,0.138971,0.058489,NaN,NaN,NaN,NaN,NaN
514,DSLXSJZSTL_r__KGJXRPELGX_f,2,7,True,2,1,True,0.693147,20,17091,...,0.0,0.0,3420.000000,0.818182,0.800000,NaN,NaN,NaN,NaN,NaN
515,GPXHVRRZLC_f__KYQOKYBCOW_f,2,8,False,2,1,True,0.693147,12,65779,...,0.0,0.0,0.000000,0.500000,0.000000,NaN,NaN,NaN,NaN,NaN
516,XXIWNZXZTK_f__ZLAJFQLBFQ_r,2,3,False,2,1,True,0.693147,6,83629,...,0.0,0.0,0.000000,0.500000,0.000000,NaN,NaN,NaN,NaN,NaN


In [25]:
jdf_singleton = jdf_all[jdf_all['singleton'] == True]
jdf_all[jdf_all['singleton_has_own_cluster'] == True]


,junction_name,n_iso,n_blocks,has_dupl,n_categories,majority_category,singleton,cat_entropy,n_nodes,min_length,...,mean_length_translocation,mean_length_inversion,mean_event_length,block_ji_dist,edge_ji_dist,has_singleton_gain,has_singleton_loss,singleton_gain_isolate,singleton_loss_isolate,singleton_has_own_cluster
24,ISADOQMCVR_r__JKXDSPORHF_r,222,3,True,2,221,True,0.028831,7,7855,...,0.0,0.0,0.0,0.997748,0.996997,True,False,NZ_CP116085.1,NaN,True
107,IFRPFFEGON_f__LSBXOCNNHO_f,222,3,False,2,221,True,0.028831,5,4258,...,0.0,0.0,0.0,0.996997,0.990991,True,False,NZ_CP107142.1,NaN,True
240,LWRCACRWJW_r__PIVHURLJVP_r,222,3,False,2,221,True,0.028831,5,9756,...,0.0,0.0,0.0,0.996997,0.990991,False,True,NaN,NZ_CP116085.1,True
418,GAKJONQISG_r__UJBGATQZSS_r,222,4,False,2,221,True,0.028831,6,2354,...,0.0,0.0,0.0,0.995495,0.990991,True,True,NZ_CP116085.1,NZ_CP116085.1,True
420,FZZJSRRGUR_r__WNLYKTZGKW_r,222,4,False,2,221,True,0.028831,6,1903,...,0.0,0.0,0.0,0.995495,0.990991,True,True,NZ_CP015074.2,NZ_CP015074.2,True
437,DGMGBEYIHM_f__JVDYVQZUBR_f,222,3,False,2,221,True,0.028831,5,10012,...,0.0,0.0,0.0,0.996997,0.990991,True,False,NZ_CP116085.1,NaN,True


In [ ]:
# Percentage of single events associated with their own cluster --> potential homologous recombination
len(jdf_all[jdf_all['singleton_has_own_cluster'] == True]) / len(jdf_all[~jdf_all["singleton_has_own_cluster"].isna()]) * 100

2.197802197802198

In [9]:
jdf_singleton[~jdf_singleton["singleton_loss_isolate"].isna()]

,junction_name,n_iso,n_blocks,has_dupl,n_categories,majority_category,singleton,cat_entropy,n_nodes,min_length,...,mean_length_deletion,mean_length_translocation,mean_length_inversion,mean_event_length,block_ji_dist,edge_ji_dist,has_singleton_gain,has_singleton_loss,singleton_gain_isolate,singleton_loss_isolate
72,HLZGFRDRSW_r__SYQZDBITDR_r,222,4,False,2,221,True,0.028831,6,40105,...,612.0,0.0,0.0,690.0,0.995495,0.990991,True,True,NZ_CP072980.1,NZ_CP072980.1
74,HLDRILYIZH_r__YSKQIEWBCG_r,222,4,False,2,221,True,0.028831,6,10985,...,841.0,0.0,0.0,804.0,0.995495,0.990991,True,True,NZ_CP051615.1,NZ_CP051615.1
94,HUIGUZLHBJ_r__YRROIHFKBD_f,222,3,False,2,221,True,0.028831,5,41243,...,608.0,0.0,0.0,608.0,0.996997,0.990991,False,True,<NA>,NZ_CP054219.1
123,KUDTBWWBJE_f__YSEMYTMKWD_f,222,4,False,2,221,True,0.028831,6,6802,...,510.0,0.0,0.0,639.0,0.995495,0.990991,True,True,NZ_CP124455.1,NZ_CP124455.1
160,QQLFXKORNH_r__TUJRNXXQZV_r,222,3,False,2,221,True,0.028831,5,8260,...,0.0,0.0,0.0,0.0,0.996997,0.990991,False,True,<NA>,NZ_CP021207.1
164,QLKLQMYUEK_f__SVGDCUWZRJ_f,222,3,False,2,221,True,0.028831,5,69126,...,0.0,0.0,0.0,0.0,0.996997,0.990991,False,True,<NA>,NZ_CP124517.1
171,PUSHEQNFCL_r__WFTVPHITVT_r,222,3,False,2,221,True,0.028831,5,12853,...,430.0,0.0,0.0,430.0,0.996997,0.990991,False,True,<NA>,NZ_CP015076.1
228,MRDJEEINGI_f__PHTUBOWZWA_r,222,4,False,2,221,True,0.028831,6,5378,...,1445.0,0.0,0.0,1106.5,0.995495,0.990991,True,True,NZ_CP029576.1,NZ_CP029576.1
230,MPNWYFWMOG_r__RNYSWPAVAU_r,222,3,False,2,221,True,0.028831,5,16888,...,358.0,0.0,0.0,358.0,0.996997,0.990991,False,True,<NA>,NZ_CP128907.1
234,MFKARJGFFB_f__UNPTZNKWVD_f,222,3,False,2,221,True,0.028831,5,21674,...,1572.0,0.0,0.0,1572.0,0.996997,0.990991,False,True,<NA>,NZ_CP095520.1


In [11]:
cluster_df = pd.read_csv("../results/consensus_analysis/cluster_maps_0_001.csv")
cluster_df

,junction_name,n_clusters,n_isolates,NZ_CP020116.1,NZ_CP028483.1,NZ_LS992190.1,NZ_CP024720.1,NZ_CP027534.1,NZ_CP096110.1,NZ_CP049077.2,...,NZ_CP124320.1,NZ_CP098203.1,NZ_CP124471.1,NZ_CP124347.1,NZ_CP124498.1,NZ_CP103465.1,NZ_CP124515.1,NZ_CP124322.1,NZ_CP095137.1,NZ_OX637959.1
0,HUTOPWFGVH_f__WJBYSSJHSE_f,2,222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
1,IZQZNHJQRQ_f__ZVPGGEJIIF_f,1,222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,JNLRGIQXOF_f__YQUHGDANHE_f,3,222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0,3.0
3,JMOMDSHCBS_r__SPDPCYMYDN_r,1,222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,JKRDVEYDGL_f__OTPJRRWJRK_f,2,222,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
543,GPXHVRRZLC_f__KYQOKYBCOW_f,2,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
544,EOBHADSLFU_f__SKPHAXSFLS_r,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0
545,XXIWNZXZTK_f__ZLAJFQLBFQ_r,1,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
546,YUOECYBHUS_f__ZTHKZYHPIX_f,1,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
example_junction = "ISADOQMCVR_r__JKXDSPORHF_r"
example_pangraph = pp.Pangraph.from_json(f"../results/junction_pangraphs/{example_junction}.json")

In [14]:
blockstats_df = example_pangraph.to_blockstats_df()
accessory_block_id = blockstats_df[blockstats_df['core'] == False].index
example_pangraph.to_blockcount_df().loc[accessory_block_id].to_numpy().sum()

np.int64(223)

In [15]:
example_pangraph.to_blockcount_df()

path_id,NZ_CP124374.1,NZ_CP018970.1,NZ_CP049077.2,NZ_CP107128.1,NZ_CP095540.1,NZ_CP024717.1,NZ_CP015074.2,NZ_CP124424.1,NZ_CP124492.1,NZ_CP097370.1,...,NZ_CP021689.1,NZ_CP124355.1,NZ_CP116145.1,NZ_CP134384.1,NZ_AP026794.1,NZ_CP103465.1,NZ_CP019015.1,NZ_CP103557.1,NZ_CP027534.1,NZ_CP059279.1
block_id,,,,,,,,,,,,,,,,,,,,,
2323961654240359251,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
4277430780794804559,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1
8368548215481589430,1,1,1,1,1,1,1,1,1,1,...,1,1,1,1,1,1,1,1,1,1


In [26]:
jdf_all[(jdf_all['singleton'] == False) & (jdf_all['n_categories'] == 2)]

,junction_name,n_iso,n_blocks,has_dupl,n_categories,majority_category,singleton,cat_entropy,n_nodes,min_length,...,mean_length_translocation,mean_length_inversion,mean_event_length,block_ji_dist,edge_ji_dist,has_singleton_gain,has_singleton_loss,singleton_gain_isolate,singleton_loss_isolate,singleton_has_own_cluster
3,JMOMDSHCBS_r__SPDPCYMYDN_r,222,3,False,2,220,False,0.051397,5,7364,...,0.0,0.0,777.0,0.994021,0.982064,NaN,NaN,NaN,NaN,NaN
20,IXLMXEMXWI_r__XRXZJDDTTM_r,222,3,False,2,218,False,0.090222,5,29821,...,0.0,0.0,776.0,0.988151,0.964453,NaN,NaN,NaN,NaN,NaN
21,IWNOJXIDQP_r__KCBEPSEUCA_f,222,3,False,2,220,False,0.051397,5,14408,...,0.0,0.0,777.0,0.994021,0.982064,NaN,NaN,NaN,NaN,NaN
25,ISADOQMCVR_f__QHGKDXJCDB_f,222,3,False,2,214,False,0.155135,5,20543,...,0.0,0.0,0.0,0.976737,0.930211,NaN,NaN,NaN,NaN,NaN
31,QXRGVLICMC_r__RQUTOCWADD_r,222,3,False,2,220,False,0.051397,5,4375,...,0.0,0.0,777.0,0.994021,0.982064,NaN,NaN,NaN,NaN,NaN
35,KLARMMLEPU_f__OJTZQVKWWG_r,222,3,False,2,220,False,0.051397,5,40780,...,0.0,0.0,1253.0,0.994021,0.982064,NaN,NaN,NaN,NaN,NaN
37,RYJVCTXEHC_r__STVEZQVFDI_r,222,3,False,2,220,False,0.051397,5,13875,...,0.0,0.0,773.0,0.994021,0.982064,NaN,NaN,NaN,NaN,NaN
64,HHRCVCXKIA_r__TJDPHITKZG_r,222,3,False,2,219,False,0.071585,5,15397,...,0.0,0.0,777.0,0.991073,0.973218,NaN,NaN,NaN,NaN,NaN
77,HKKMVAAVLK_f__YXFAUSIPGJ_r,222,3,False,2,173,False,0.527819,5,4744,...,0.0,0.0,0.0,0.884812,0.654437,NaN,NaN,NaN,NaN,NaN
103,IHERCMJOSU_f__JJRRWBDVGH_f,222,3,False,2,199,False,0.332930,5,26445,...,0.0,0.0,1446.0,0.937807,0.813420,NaN,NaN,NaN,NaN,NaN
